# Visual Health — formats d'image APTOS (CPU)

Relève la taille d'origine (largeur × hauteur) des 3 662 images annotées
d'APTOS 2019, indicateur de l'appareil photo. Sert à vérifier en local
(`scripts/aptos_device_check.py`) si le diagnostic d'APTOS est lié à
l'appareil, et si les têtes gardent leurs performances à appareil égal.

**Pas besoin de GPU** : `Exécution > Modifier le type d'exécution > CPU`
(aucun quota GPU consommé). Les tailles sont lues dans l'en-tête des images,
directement dans l'archive, sans la décompresser.

**Avant de lancer (Colab)** :
1. Règlement de la compétition
   [aptos2019-blindness-detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection)
   déjà accepté.
2. Secret 🔑 `KAGGLE_API_TOKEN` (clé `KGAT_…`), accès activé pour ce notebook.
3. `Exécution > Tout exécuter` (~5 min). `aptos_formats.csv` se télécharge à
   la fin.

Sur Kaggle : ajouter la compétition en input, pas de secret nécessaire.

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
os.chdir(WORK)
ZIP = None
if not ON_KAGGLE:
    from google.colab import userdata
    token = userdata.get("KAGGLE_API_TOKEN").strip()  # jamais écrit dans le notebook
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
        f.write(token)
    os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    os.environ["KAGGLE_API_TOKEN"] = token
    ZIP = f"{WORK}/aptos2019-blindness-detection.zip"
    if not os.path.exists(ZIP):
        !pip install -q -U kaggle
        !kaggle competitions download -c aptos2019-blindness-detection -p {WORK} -q
    print(ZIP, f"{os.path.getsize(ZIP) / 1e9:.1f} Go")

In [ ]:
import csv
import glob
import zipfile

from PIL import Image

rows = []
if ON_KAGGLE:
    root = sorted(glob.glob("/kaggle/input/**/train_images", recursive=True))[0]
    for path in sorted(glob.glob(f"{root}/*.png")):
        with Image.open(path) as image:
            rows.append((os.path.basename(path)[:-4], image.width, image.height))
else:
    with zipfile.ZipFile(ZIP) as archive:
        members = sorted(n for n in archive.namelist() if n.startswith("train_images/") and n.endswith(".png"))
        for name in members:
            with archive.open(name) as f, Image.open(f) as image:  # lit seulement l'en-tête
                rows.append((os.path.basename(name)[:-4], image.width, image.height))
with open("aptos_formats.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id_code", "width", "height"])
    writer.writerows(rows)
print(len(rows), "images ;", len({(w, h) for _, w, h in rows}), "formats")
assert len(rows) == 3662, "nombre d'images inattendu"

In [ ]:
if not ON_KAGGLE:
    from google.colab import files
    files.download("aptos_formats.csv")